# LLM Concepts

**Learning objectives:** explain tokenization, token IDs, one-hot vectors, embeddings, Word2Vec, sequence context, attention, and positional information.

## Exercise 1 — Tokenize a normal sentence
Tokenization separates text into units; IDs identify tokens in a vocabulary. Neither step provides semantics by itself.

In [ ]:
sentence = "the dog runs fast"
#sentence = "the dog runs and the cat sleeps"
tokens = sentence.lower().split()
vocab = {w:i for i,w in enumerate(sorted(set(tokens)))}
ids = [vocab[w] for w in tokens]
print('Tokens:',tokens)
print('Vocabulary:',vocab)
print('IDs:',ids)
assert len(tokens)==len(ids)

**Your turn:** Change `sentence` to `"the dog runs and the cat sleeps"`. What does the ID tell us about meaning?

## Exercise 2 — One-hot vectors versus embedding lookup
One-hot vectors encode identity. In an embedding lookup, token IDs index rows of a **trainable** table. The random table below has no learned meaning yet.

In [ ]:
import numpy as np
np.random.seed(3)
V=len(vocab)
onehot = np.eye(V)[ids]
embedding_table=np.random.normal(size=(V,4)).round(2)
lookup=embedding_table[ids]
print('One-hot shape:',onehot.shape,'  Embedded shape:',lookup.shape)
print('Token:',tokens[1], '\nOne-hot:',onehot[1], '\nEmbedding:',lookup[1])


**Questions:** Which representation is sparse?

## Where does Word2Vec fit?
**Word2Vec is an early neural method for learning word embeddings from neighboring words.** It uses prediction objectives such as Skip-gram or CBOW. It is **not** a transformer or a full LLM. It learns a largely **static** vector for each word: *bank* has the same Word2Vec lookup regardless of which sentence it appears in. Transformer layers instead produce **context-dependent** representations.

Workflow: **Text → tokenization → IDs → Word2Vec training → trained word vectors**. These vectors can then be used by other models, including RNNs.

The following exercise trains a miniature Word2Vec model on a tiny synthetic corpus. Such small data are for mechanics, not semantic accuracy.

## Exercise 3 — Train Word2Vec on example phrases
You can expand the corpus. This exercise uses `gensim`, installed in Colab if needed.

In [ ]:
# Uncomment this in Colab if gensim is not installed or incompatible:
%pip -q install -U gensim scipy
try:
    from gensim.models import Word2Vec
    corpus = [
      'dog puppy animal runs fast', 'cat kitten animal sleeps',
      'dog puppy animal plays', 'cat kitten animal plays',
      'plant leaf photosynthesis light', 'plant root absorbs water',
      'leaf plant absorbs light', 'puppy dog runs', 'kitten cat sleeps'
    ]
    sentences=[s.split() for s in corpus]*60
    w2v=Word2Vec(sentences,vector_size=16,window=2,min_count=1,sg=1,seed=1,workers=1,epochs=30)
    print('dog vector:',w2v.wv['dog'][:5].round(3))
    print('Closest words to dog:',w2v.wv.most_similar('dog',topn=3))
except ImportError:
    print('Install gensim using the command above, then rerun this cell.')

**Questions:** What information did the training sentences contribute? If a word has two meanings, does this small Word2Vec model make two different vectors automatically? Explain why similarity from a tiny corpus should not be overinterpreted.

## Exercise 4 — Why the context matters: “bank”
Use two sentences with the same token. This simplified illustration mimics the difference between **static lookup** and **contextualized representation**; it is not a real transformer.

In [ ]:
sentence1='money at the bank'
sentence2='water close to the bank'
static_bank=np.array([0.2,0.3,0.6])
context1=static_bank+np.array([0.5,0.0,-0.2]) # illustrative contextual update
context2=static_bank+np.array([-0.2,0.5,0.1])
print('Static bank vector (both sentences):',static_bank)
print('Contextual version near money:',context1)
print('Contextual version near water:',context2)
print('Different contextual vectors:',not np.allclose(context1,context2))